In [2]:
"""
Step 2 of 5 — aggregate each country-year with the OWAGDP operator.

Self-contained: it asks you to upload panel.csv when you run it, and downloads
its outputs when it finishes. Nothing has to survive from a previous cell.

UPLOAD   panel.csv        iso, year, IMF, OECD, EC     (from 01_build_panel.py)
WRITES   aggregates.csv   panel + owa_0.1 ... owa_0.9, mean, spread, range, roles
         weights.csv      Table I, the maximum-entropy weight vectors
"""
import os
import numpy as np
import pandas as pd

INST = ["IMF", "OECD", "EC"]
GRID = [round(0.1 * i, 1) for i in range(1, 10)]
A_LO, A_HI = 0.2, 0.8          # the pair reported as the attitudinal spread
ROUND = 4                      # applied ONLY on write, never before arithmetic

REQUIRED = {"iso", "year", *INST}

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_panel():
    """Any csv carrying iso, year, IMF, OECD, EC counts, whatever it is named."""
    for f in sorted(os.listdir(".")):
        if f.lower().endswith(".csv"):
            try:
                if REQUIRED.issubset(pd.read_csv(f, nrows=1).columns):
                    return f
            except Exception:
                pass
    return None


name = find_panel()
if name is None:
    if IN_COLAB:
        print("Upload panel.csv")
        files.upload()
        name = find_panel()
    if name is None:
        raise FileNotFoundError(
            "no csv here with columns iso, year, IMF, OECD, EC.\n"
            f"Present: {sorted(os.listdir('.'))}")
print(f"panel: {name}\n")


def me_weights(alpha, n=3):
    """Maximum-entropy OWA weights for a declared orness.
    Geometric solution w_i ~ h^(i-1); orness is strictly decreasing in h."""
    if abs(alpha - 0.5) < 1e-12:
        return np.full(n, 1.0 / n)

    def orness(h):
        w = h ** np.arange(n)
        w = w / w.sum()
        return ((n - 1 - np.arange(n)) * w).sum() / (n - 1), w

    lo, hi = 1e-9, 1e9
    for _ in range(200):
        h = np.sqrt(lo * hi)
        lo, hi = (h, hi) if orness(h)[0] > alpha else (lo, h)
    return orness(np.sqrt(lo * hi))[1]


p = pd.read_csv(name)
G = p[INST].values
S = np.sort(G, axis=1)[:, ::-1]            # descending: g_(1) ... g_(n)

for a in GRID:                              # OWAGDP across the orness grid
    p[f"owa_{a}"] = S @ me_weights(a)

# every derived column is computed from RAW values, then rounded once on write
p["mean"] = G.mean(axis=1)
p["spread"] = p[f"owa_{A_HI}"] - p[f"owa_{A_LO}"]
p["range"] = S[:, 0] - S[:, -1]
p["most_opt"] = [INST[i] for i in G.argmax(axis=1)]
p["most_pes"] = [INST[i] for i in G.argmin(axis=1)]

p.round(ROUND).to_csv("aggregates.csv", index=False)
pd.DataFrame([me_weights(a) for a in GRID], index=GRID,
             columns=["w1", "w2", "w3"]).round(3).to_csv("weights.csv",
                                                         index_label="alpha")

# ---------------------------------------------------------------- report ----
print(f"{p.iso.nunique()} economies x 2 years = {len(p)} cells -> aggregates.csv\n")

print("Table I — maximum-entropy weights")
for a in GRID:
    print(f"  alpha {a}: {np.round(me_weights(a), 3)}")

print("\nTable II")
for iso in ("SWE", "DNK", "ITA"):
    for y in (2026, 2027):
        r = p[(p.iso == iso) & (p.year == y)]
        if r.empty:
            continue
        r = r.iloc[0]
        print(f"  {iso} {y}  {r.IMF:6.3f} {r.OECD:6.3f} {r.EC:6.3f} | "
              f"mean {r['mean']:6.3f}  a={A_LO} {r[f'owa_{A_LO}']:6.3f}  "
              f"a={A_HI} {r[f'owa_{A_HI}']:6.3f}  spread {r.spread:.3f}")

print("\nSection V-B")
print(f"  spread  median {p.spread.median():.3f} | mean {p.spread.mean():.3f} "
      f"| IQR {p.spread.quantile(.25):.3f}-{p.spread.quantile(.75):.3f}")
print(f"  cells with spread below 0.05: {(p.spread < 0.05).sum()}")
i = p.spread.idxmax()
print(f"  largest spread {p.spread.max():.3f}  ({p.loc[i,'iso']} {p.loc[i,'year']})")
for k in INST:
    print(f"  {k:<4} most optimistic {(p.most_opt==k).sum():3d} | "
          f"most pessimistic {(p.most_pes==k).sum():3d}")
t = p.pivot(index="iso", columns="year", values="most_opt")
b = p.pivot(index="iso", columns="year", values="most_pes")
yrs = sorted(t.columns)
print(f"  role changes between the two years: "
      f"{((t[yrs[0]]!=t[yrs[1]])|(b[yrs[0]]!=b[yrs[1]])).sum()} of {p.iso.nunique()}")

# at n = 3 the middle term of the spread cancels, so the spread is a fixed
# multiple of the range -- state this in the paper rather than let a referee
# derive it
k = me_weights(A_HI)[0] - me_weights(A_HI)[-1]
print(f"\n  spread == {k:.3f} x range exactly "
      f"(max deviation {abs(p.spread - k*p['range']).max():.2e})")

if IN_COLAB:
    files.download("aggregates.csv")
    files.download("weights.csv")

Upload panel.csv


Saving panel.csv to panel.csv
panel: panel.csv

37 economies x 2 years = 74 cells -> aggregates.csv

Table I — maximum-entropy weights
  alpha 0.1: [0.026 0.147 0.826]
  alpha 0.2: [0.082 0.236 0.682]
  alpha 0.3: [0.154 0.292 0.554]
  alpha 0.4: [0.238 0.323 0.438]
  alpha 0.5: [0.333 0.333 0.333]
  alpha 0.6: [0.438 0.323 0.238]
  alpha 0.7: [0.554 0.292 0.154]
  alpha 0.8: [0.682 0.236 0.082]
  alpha 0.9: [0.826 0.147 0.026]

Table II
  SWE 2026   1.963  1.899  1.760 | mean  1.874  a=0.2  1.810  a=0.8  1.931  spread 0.122
  SWE 2027   1.906  2.544  2.170 | mean  2.207  a=0.2  2.021  a=0.8  2.403  spread 0.383
  DNK 2026   2.000  2.548  1.910 | mean  2.153  a=0.2  1.984  a=0.8  2.366  spread 0.383
  DNK 2027   1.550  1.486  1.753 | mean  1.596  a=0.2  1.523  a=0.8  1.683  spread 0.160
  ITA 2026   0.521  0.530  0.545 | mean  0.532  a=0.2  0.525  a=0.8  0.540  spread 0.015
  ITA 2027   0.500  0.569  0.596 | mean  0.555  a=0.2  0.524  a=0.8  0.582  spread 0.058

Section V-B
  spread  m

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>